# 46. A joint hyperparameter search, which this repo has never run

**One variable against ledger row 93** (`xgb_te_fe`, CV 0.968005): the hyperparameter vector.
Same 49 features, same encoder fingerprinted `0642e41750ef8bab`, same ratio block, same folds,
same seed, same `learning_rate` and tree budget.

## Correcting something this repo has been telling itself

`NOTES.md` has been saying the tuning axis is exhausted, on the strength of six sweeps that all
came back null: `num_leaves`, encoder smoothing, inner splits, `max_depth`, `max_bin`, CatBoost
iterations. Every one of those was **one parameter at a time**.

Hyperparameters interact. A one-at-a-time sweep finds the best value of each knob *holding the
others at their current values*, which says nothing about whether a different combination is
better. Six null sweeps are evidence that this configuration is a local optimum along six axes.
They are not evidence that it is a good optimum.

**And three of the four regularisation parameters on our best model have never been chosen at
all.** They are library defaults:

| parameter | ours | srcC's public starter |
|---|---|---|
| `min_child_weight` | **1, the default** | 15 |
| `reg_lambda` | **1, the default** | 3.0 |
| `reg_alpha` | **0, the default** | 0.05 |
| `gamma` | **0, the default** | 0 |

On 691,369 rows, `min_child_weight=1` permits a leaf built from a single row. That is not a
tuned choice, it is an untouched one, and the depth sweep in rows 62 to 65 was conducted with it
sitting there the whole time.

`NOTES.md` says not to tune before the feature set is stable. It is now stable: the encoder has
not moved since row 17 and the ratio block settled in row 93. The precondition is met.

The public measurement: kito_pl reports **+0.0010 OOF from 26 Optuna trials** on this competition,
second only to `max_bin` in their table. Our `max_bin` came back null because our encoder already
did that job. There is no equivalent reason for the regularisation parameters to be pre-empted.

## The design, and where its cost goes

A full five-fold evaluation of one configuration costs about nine minutes, so forty of them would
be six hours before any validation. Two changes make it affordable:

1. **The encoded frames are built once and cached.** The encoder is a function of the fold and
   the data, not of the hyperparameters, so rebuilding it per trial is pure waste. Row 93 spent a
   meaningful part of its nine minutes encoding.
2. **The search scores on folds 0 and 1 only.** The final validation is the full five.

## The optimism this introduces, measured rather than waved away

Selecting a configuration on folds 0 and 1 and then reporting its five-fold CV means two of the
five folds helped choose it. That is the same bias every hyperparameter sweep in this ledger
carries, and row 32 handled it the right way: it reported `top9` over the folds that did not
select it and printed the gap.

This notebook does the same. It reports the winner's **full five-fold CV** and its **CV over folds
2, 3 and 4 alone**, which are untouched by the search, and prints the difference. The clean number
is the one to believe.

## The search space

| parameter | range | ours now |
|---|---|---|
| `max_depth` | 4 to 10 | 6 |
| `min_child_weight` | 1 to 64, log | 1 |
| `subsample` | 0.6 to 1.0 | 0.8 |
| `colsample_bytree` | 0.4 to 1.0 | 0.8 |
| `reg_alpha` | 1e-3 to 10, log | 0 |
| `reg_lambda` | 0.1 to 20, log | 1 |
| `gamma` | 1e-3 to 5, log | 0 |

`learning_rate` stays at 0.05 and `n_estimators` at 2000, deliberately. Searching them would make
every trial a different cost and would break the budget convention every row in this ledger shares.
It also keeps this closer to one variable: the regularisation geometry moves, the budget does not.

## The prediction, written before the run

**+0.0004 to +0.0010 on the full five folds**, with the clean folds-2-to-4 figure coming in
somewhat lower than the headline, and `min_child_weight` moving furthest from its default.

**The honest case against**, which has beaten my predictions three runs running. Rows 62 to 65
swept depth and found 6 optimal with the curve falling on both sides and losing every fold at
every other value. That is a fairly emphatic local optimum, and a joint search may simply
rediscover it. If the search returns something within noise of row 93, the correct reading is that
the six null sweeps were measuring a genuinely good configuration rather than a lucky one, and the
tuning axis really is closed.

## What this decides

Nothing about the stack. The winner is a new member vector and membership is a separate row. No
submission csv.

In [ ]:
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0
TARGET = "addicted_label"

# Row 93's budget, held fixed. Only the regularisation geometry is searched.
LR = 0.05
N_EST = 2000
PROBE_FOLD = 0
THREADS = -1

# Folds the search is allowed to see. The rest are held back so the winner has a figure
# that no part of the search influenced.
SEARCH_FOLDS = [0, 1]
CLEAN_FOLDS = [2, 3, 4]

N_TRIALS = 40

# Row 93's configuration, which is trial 0 so the search starts from what we already have.
BASE_PARAMS = dict(max_depth=6, min_child_weight=1, subsample=0.8,
                   colsample_bytree=0.8, reg_alpha=0.0, reg_lambda=1.0, gamma=0.0)

BASELINE_NAME, BASELINE_CV, BASELINE_ROW = "xgb_te_fe", 0.968005, "row 93 xgb_te_fe"
MAX_HOURS = 9.0
EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_ENCODER_FP = "0642e41750ef8bab"
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

print(f"SMOKE = {SMOKE}   trials {N_TRIALS}   search folds {SEARCH_FOLDS}")
print(f"held back for the clean estimate: {CLEAN_FOLDS}")

## Stage 1. Data, folds, leak checklist

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError as e:
    raise SystemExit(f"optuna is required and not present: {e}")

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")
print(f"xgboost {xgb.__version__}, optuna {optuna.__version__}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, N_TRIALS = 200, 6
    THREADS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
X = train[COLS].copy()
X_test = test[COLS].copy()

folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i
sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print(f"\nrows {len(train):,}   fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
print("SMOKE: sha EXPECTED to differ, not a check." if SMOKE else
      ("fold alignment: VERIFIED" if ALIGNED else "fold alignment: MISMATCH"))

## Stage 2. The encoder and the ratio block, both from row 93

In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

In [ ]:
DST, SM, GM, WS = ("daily_screen_time_hours", "social_media_hours",
                   "gaming_hours", "work_study_hours")
SLP, WKD = "sleep_hours", "weekend_screen_time"
NOTIF, OPENS = "notifications_per_day", "app_opens_per_day"

RATIO_COLS = ["component_total", "slack", "weekend_lift", "weekend_ratio",
              "social_share", "gaming_share", "work_share", "sleep_minus_screen",
              "screen_to_sleep", "opens_per_hour", "notif_per_hour",
              "notif_per_open", "engagement"]


def safe_div(a, b):
    b = b.replace(0, np.nan)
    return a / b


def ratio_block(df):
    o = pd.DataFrame(index=df.index)
    o["component_total"] = df[[SM, GM, WS]].sum(axis=1, min_count=3)
    o["slack"] = df[DST] - o["component_total"]
    o["weekend_lift"] = df[WKD] - df[DST]
    o["weekend_ratio"] = safe_div(df[WKD], df[DST])
    o["social_share"] = safe_div(df[SM], df[DST])
    o["gaming_share"] = safe_div(df[GM], df[DST])
    o["work_share"] = safe_div(df[WS], df[DST])
    o["sleep_minus_screen"] = df[SLP] - df[DST]
    o["screen_to_sleep"] = safe_div(df[DST], df[SLP])
    o["opens_per_hour"] = safe_div(df[OPENS], df[DST])
    o["notif_per_hour"] = safe_div(df[NOTIF], df[DST])
    o["notif_per_open"] = safe_div(df[NOTIF], df[OPENS])
    o["engagement"] = df[NOTIF] + df[OPENS]
    return o.replace([np.inf, -np.inf], np.nan).astype(np.float64)


rng = np.random.default_rng(0)
_perm = train.copy()
_perm[TARGET] = rng.permutation(train[TARGET].to_numpy())
BLOCK_OK = bool(ratio_block(_perm).equals(ratio_block(train))
                and list(ratio_block(train).columns) == RATIO_COLS)
print(f"ratio block is a pure function of the features, not of y: {BLOCK_OK}")
del _perm
gc.collect()


def build_arm(fold, want_test=False):
    """Row 93's 49-column frame."""
    tr = np.where(folds != fold)[0]
    va = np.where(folds == fold)[0]
    Xtr, Xva, Xte = build(X, y, tr, va, X_test if want_test else None)
    for c in CAT:
        Xtr[c] = Xtr[c].astype("category")
        Xva[c] = Xva[c].astype("category")
    Xtr = pd.concat([Xtr, ratio_block(train.iloc[tr]).reset_index(drop=True)], axis=1)
    Xva = pd.concat([Xva, ratio_block(train.iloc[va]).reset_index(drop=True)], axis=1)
    if want_test:
        for c in CAT:
            Xte[c] = Xte[c].astype("category")
        Xte = pd.concat([Xte, ratio_block(test).reset_index(drop=True)], axis=1)
    return tr, va, Xtr, Xva, Xte


# The encoder depends on the fold and the data, never on the hyperparameters, so it is
# built once here and reused by every trial. Rebuilding it per trial would be the single
# largest waste in this notebook.
t0 = time.time()
CACHE = {}
for f in SEARCH_FOLDS:
    tr, va, Xtr, Xva, _ = build_arm(f)
    CACHE[f] = (tr, va, Xtr, Xva)
print(f"cached encoded frames for folds {SEARCH_FOLDS} in {time.time() - t0:.0f}s, "
      f"{CACHE[SEARCH_FOLDS[0]][2].shape[1]} columns")

## Stage 3. Bench, and how many trials fit in the budget

In [ ]:
def make_xgb(p, n_est=None):
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True,
        learning_rate=LR, n_estimators=n_est or N_EST,
        random_state=SEED, n_jobs=THREADS, verbosity=0, **p)


def score_folds(p, fold_list, cache=None):
    aucs = []
    for f in fold_list:
        if cache is not None and f in cache:
            tr, va, Xtr, Xva = cache[f]
        else:
            tr, va, Xtr, Xva, _ = build_arm(f)
        m = make_xgb(p).fit(Xtr, y[tr])
        aucs.append(float(roc_auc_score(y[va], m.predict_proba(Xva)[:, 1])))
        del m
        gc.collect()
    return np.array(aucs)


def hhmm(s):
    s = int(s)
    return f"{s // 3600}h {s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m {s % 60:02d}s"


t0 = time.time()
_ = score_folds(BASE_PARAMS, [SEARCH_FOLDS[0]], CACHE)
per_fold = time.time() - t0
trial_cost = per_fold * len(SEARCH_FOLDS)
final_cost = per_fold * 5 * 2          # baseline and winner, full five folds each
projected = trial_cost * N_TRIALS + final_cost
print(f"one fold at row 93's config: {hhmm(per_fold)}")
print(f"one trial ({len(SEARCH_FOLDS)} folds):    {hhmm(trial_cost)}")
print(f"{N_TRIALS} trials:                {hhmm(trial_cost * N_TRIALS)}")
print(f"final validation, 2 configs x 5 folds: {hhmm(final_cost)}")
print(f"projected TOTAL:            {hhmm(projected)}")
GO = projected < MAX_HOURS * 3600 or SMOKE
print("within budget" if GO else
      f"OVER the {MAX_HOURS}h guard. Lower N_TRIALS and re-push.")

## Stage 4. The search

In [ ]:
assert LEAK_OK and CLEAN, "leak checks failed"
assert ENCODER_MATCH, "encoder does not match 13"
assert BLOCK_OK, "the ratio block is not a pure function of the features"
assert GO, "over the time guard"
if not SMOKE:
    assert ALIGNED, "fold sha mismatch"


def objective(trial):
    p = dict(
        max_depth=trial.suggest_int("max_depth", 4, 10),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 64.0, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
        gamma=trial.suggest_float("gamma", 1e-3, 5.0, log=True),
    )
    return float(score_folds(p, SEARCH_FOLDS, CACHE).mean())


sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
# Trial 0 is row 93's own configuration, so the search starts from what we already have
# and can never return something worse than the incumbent.
# reg_alpha and gamma are searched on a log scale, which cannot represent row 93's literal
# 0.0, so the seed trial uses the lower bound instead. At 1e-3 on 691,369 rows that is
# numerically indistinguishable from no penalty. BASE_PARAMS keeps the true 0.0 values and
# is what the full five-fold validation below reproduces row 93 with.
SEED_TRIAL = {k: (1e-3 if (k in ("reg_alpha", "gamma") and v == 0.0) else v)
              for k, v in BASE_PARAMS.items()}
study.enqueue_trial(SEED_TRIAL)

t0 = time.time()
done = [0]


def cb(st, tr):
    done[0] += 1
    if done[0] % 5 == 0 or done[0] == 1:
        el = time.time() - t0
        print(f"  trial {done[0]:>3}/{N_TRIALS}  best {st.best_value:.6f}  "
              f"elapsed {hhmm(el)}, about {hhmm(el / done[0] * (N_TRIALS - done[0]))} left")


study.optimize(objective, n_trials=N_TRIALS, callbacks=[cb])
print(f"\nsearch done in {hhmm(time.time() - t0)}")

# The seed trial is row 93's config with reg_alpha and gamma at 1e-3 rather than the
# literal 0.0 the log scale cannot express, so it is not exactly the incumbent. Score the
# true incumbent separately and let it win ties, otherwise the search can "win" by a
# millionth with a configuration that is really just row 93 plus rounding.
base_search = float(score_folds(BASE_PARAMS, SEARCH_FOLDS, CACHE).mean())
print(f"row 93's config on the search folds: {base_search:.6f}")
print(f"best found on the search folds:      {study.best_value:.6f}  "
      f"({study.best_value - base_search:+.6f})")

SEARCH_WON = study.best_value > base_search
if not SEARCH_WON:
    print()
    print("THE SEARCH FOUND NOTHING BETTER THAN THE INCUMBENT on the search folds.")
    print("The tuned arm below is therefore row 93's own configuration, the comparison is")
    print("trivially zero, and the honest reading is that the tuning axis is closed.")
print("\nbest parameters, against ours:")
for k, v in BASE_PARAMS.items():
    b = study.best_params[k]
    flag = "  <-- moved" if abs(b - v) > 1e-9 else ""
    print(f"  {k:20} {v:>10.4f} -> {b:>10.4f}{flag}")

## Stage 5. Full five-fold validation, and the clean estimate

In [ ]:
BEST = dict(study.best_params) if SEARCH_WON else dict(BASE_PARAMS)
del CACHE
gc.collect()

full = {}
oofs, tests = {}, {}
for tag, p in (("row93", BASE_PARAMS), ("tuned", BEST)):
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    per = []
    t0 = time.time()
    for f in range(5):
        tr, va, Xtr, Xva, Xte = build_arm(f, want_test=True)
        m = make_xgb(p).fit(Xtr, y[tr])
        oof[va] = m.predict_proba(Xva)[:, 1]
        tst[f] = m.predict_proba(Xte)[:, 1]
        per.append(float(roc_auc_score(y[va], oof[va])))
        del m, Xtr, Xva, Xte
        gc.collect()
    full[tag] = np.array(per)
    oofs[tag], tests[tag] = oof, tst.mean(axis=0)
    print(f"{tag:6} CV {np.mean(per):.6f} +/- {np.std(per):.6f}  [{hhmm(time.time() - t0)}]")

repro = full["row93"].mean() - BASELINE_CV
print(f"\nreproduction of {BASELINE_ROW}: {full['row93'].mean():.6f} vs {BASELINE_CV:.6f}"
      f"  delta {repro:+.2e}")
if SMOKE:
    print("SMOKE: subsampled, the reproduction check DID NOT RUN.")
else:
    print("REPRODUCED" if abs(repro) < 1e-4 else "FAILED - every number below is void")

d = full["tuned"] - full["row93"]
sd = d.std(ddof=1)
print(f"\nFULL five folds:  {full['tuned'].mean():.6f}, paired {d.mean():+.6f}, "
      f"sd {sd:.6f}, {int((d > 0).sum())}/5 folds"
      + (f", t(4)={d.mean() / (sd / np.sqrt(5)):.2f}" if sd > 0 else ""))

ci = [i for i in CLEAN_FOLDS]
dc = full["tuned"][ci] - full["row93"][ci]
print(f"CLEAN folds {CLEAN_FOLDS}, which the search never saw:")
print(f"  tuned {full['tuned'][ci].mean():.6f}, row 93 {full['row93'][ci].mean():.6f}, "
      f"paired {dc.mean():+.6f}, {int((dc > 0).sum())}/{len(ci)} folds")
print(f"  selection optimism, full minus clean: {d.mean() - dc.mean():+.6f}")
print("  THE CLEAN NUMBER IS THE ONE TO BELIEVE. Row 32 reported its arm the same way.")

In [ ]:
pre = "SMOKE_" if SMOKE else ""
np.save(OUT / f"{pre}xgb_tuned_oof.npy", oofs["tuned"])
np.save(OUT / f"{pre}xgb_tuned_test.npy", tests["tuned"])
print(f"wrote {pre}xgb_tuned_oof.npy, {pre}xgb_tuned_test.npy")

import json as _json
(OUT / f"{pre}xgb_tuned_params.json").write_text(_json.dumps(BEST, indent=2))
print(f"wrote {pre}xgb_tuned_params.json")

print("\nledger lines:")
print(f"  name    xgb_tuned\n  cv_mean {full['tuned'].mean():.6f}"
      f"\n  cv_std  {full['tuned'].std():.6f}")
print(f"\n  leak checks {'PASS' if (LEAK_OK and CLEAN) else 'FAILED'}, "
      f"encoder {EXPECTED_ENCODER_FP if ENCODER_MATCH else 'MISMATCH'}, "
      f"fold alignment {'verified' if ALIGNED else 'MISMATCH'}")
print("\nNo submission csv. Membership is a separate notebook and a separate ledger row.")